In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import plotly.express as px

session_df = pd.read_csv("../data/analysis_table/session_table.csv")
user_df = pd.read_csv("../data/analysis_table/user_table.csv")
funnel = pd.read_csv("../data/analysis_table/funnel_table.csv")

In [ ]:
# 1. Conversion Rate
view_sessions = funnel.loc[funnel['stage'] == 'view', 'sessions'].iloc[0]
funnel['conversion_rate'] = funnel['sessions'] / view_sessions

print(funnel)

In [ ]:
# Visualization: Funnel with Conversion Rate
plt.figure()
bars = plt.bar(funnel['stage'], funnel['sessions'], width = 0.4)

for index, row in funnel.iterrows():
    plt.text(x = row['stage'], y = row['sessions'], s = f"{row['conversion_rate']:.2%}", ha = 'center')

plt.title('Funnel with Conversion Rate')
plt.xlabel('Funnel Stage')
plt.ylabel('Sessions')

plt.show()

In [ ]:
# 2. Drop-off Rate
# calculate conversion rate of each steps
funnel['step_conversion_rate'] = (funnel['sessions'] / funnel['sessions'].shift(1))

# set the first step(NaN) to 1
funnel.loc[0, 'step_conversion_rate'] = 1 

# calculate drop-off rate
funnel['drop_off_rate'] = 1 - funnel['step_conversion_rate']

print(funnel)

In [ ]:
# Visualization: Drop-off rate
stages = funnel['stage']
values = funnel['sessions']
max_value = values.iloc[0]   # default view being the max value
left = (max_value - values) / 2


plt.figure(figsize = (6, 4))

plt.barh(y = stages, width = values, left = left)

for i, v in enumerate(values):
    plt.text(max_value/2, i, f"{v:,}", 
             ha='center', va='center')

plt.title("Conversion Funnel")
plt.xlabel("Number of Sessions")
plt.gca().invert_yaxis()   # put view to the top

plt.show()

In [ ]:
fig = px.funnel(
    funnel,
    x="sessions",
    y="stage"
)

fig.update_layout(
    title="Conversion Funnel"
)

fig.show()

In [ ]:
# 3. Time to Convert: how much time does viewing to purchasing takes
oct_2019 = pd.read_csv("../data/analysis_table/2019_oct_cleaned.csv")

view_times = (
    oct_2019[oct_2019['event_type'] == 'view'].groupby(['user_id', 'user_session'])['event_time']
    .min()   # take the min as first view time
    .reset_index()
    .rename(columns = {'event_time' : "first_view_time"})
)

view_times.head()

In [ ]:
purchase_times = (
    oct_2019[oct_2019['event_type'] == 'purchase'].groupby(['user_id', 'user_session'])['event_time']
    .min()   # take the min as first view time
    .reset_index()
    .rename(columns = {'event_time' : "first_purchase_time"})
)

time_to_convert = pd.merge(
    view_times, purchase_times, on = ['user_id', 'user_session'], how = 'inner'  # only considering sessions which had purchase
)

time_to_convert.head()

In [ ]:
time_to_convert.dtypes
# after the merge, time format became object again, so we need to convert to datetime again

In [ ]:
# convert to datetime again
time_to_convert['first_view_time'] = pd.to_datetime(time_to_convert['first_view_time'], utc = True)

time_to_convert['first_purchase_time'] = pd.to_datetime(time_to_convert['first_purchase_time'], utc = True)

# calculate the time to convert in minutes
time_to_convert['time_to_purchase_min'] = (time_to_convert['first_purchase_time'] - time_to_convert['first_view_time']).dt.total_seconds()/60

time_to_convert['time_to_purchase_min'] = time_to_convert['time_to_purchase_min'].round(2)

time_to_convert.head()

In [ ]:
# 4. Behavioral Difference and Uplift Ratio between converters (those who purchased) v.s. non-converters (not purchased)

# identify if the session converts or not
session_df['is_converter'] = (session_df['has_purchase'] == 1)

# add session duration time (in mins)
session_df['session_start'] = pd.to_datetime(session_df['session_start'], utc = True) # ensure datetime format
session_df['session_end'] = pd.to_datetime(session_df['session_end'], utc = True) # ensure datetime format
session_df['session_duration_sec'] = (session_df['session_end'] - session_df['session_start']).dt.total_seconds()
session_df['session_duration_min'] = (session_df['session_duration_sec'] / 60).round(2)

# define behavior metrics
behavior_metrics = ['session_duration_min', 'total_events', 'view_count', 'cart_count']

session_df.head()

In [ ]:
# calculate average of converters v.s. non-converter
behavior_summary = (session_df.groupby('is_converter')[behavior_metrics].mean().round(2))

behavior_summary = behavior_summary.T
behavior_summary.columns = ['non_converter','converter']

print(behavior_summary)

In [ ]:
# calculate Uplift Ratio to better understand the relative difference between converters v.s. non-converters
behavior_summary['uplift_ratio'] = (behavior_summary['converter'] / behavior_summary['non_converter']).round(2)

print(behavior_summary)